# 🟡 KaizenStat — Intermediate Demo (15 min)

**Level:** Intermediate | **Time:** ~15 minutes | **Dataset:** Titanic

Full 8-step pipeline + debug + trust score + auto_improve.

In [ ]:
!pip install git+https://github.com/kaizenstat-python/KaizenStat.git@92cb29bde42cfd73a797179f868b8adf52c28e84 -q
print("KaizenStat v0.5.2 installed")

## Setup — Load and Fit

`load()` handles any file type or URL. `fit()` auto-detects task type and drops ID/free-text columns.

In [ ]:
from kaizenstat import DataDoctor

doctor = DataDoctor()
doctor.load("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
doctor.fit(target="Survived")
print("Mode:", doctor.mode())

In [ ]:
health = doctor.health()
print(f"Health Score: {health.score} / 100")

In [ ]:
validation = doctor.validate()
print(f"Issues found: {len(validation.issues)}")

In [ ]:
# Preview changes before applying
doctor.fix(safe=True, preview_only=True)

In [ ]:
fixed_df = doctor.fix(safe=True)
print(f"Missing after fix: {fixed_df.isnull().sum().sum()}")

In [ ]:
train_result = doctor.train(cv=5)
print(f"Best model:  {train_result.model_name}")
print(f"Test score:  {train_result.test_score:.4f}")
print(f"Gap:         {train_result.train_score - train_result.test_score:.4f}")

### debug_model()

Answers: *Is the problem the data or the model?*
Shows feature importances and which subgroups fail most often.

In [ ]:
debug_result = doctor.debug_model()
gap = debug_result.gap
print(f"Train: {debug_result.train_score:.4f}  Test: {debug_result.test_score:.4f}  Gap: {gap:.4f}")
if gap > 0.15:
    print("OVERFITTING — try regularisation or more data")
elif gap < -0.05:
    print("UNDERFITTING — try a more complex model")
else:
    print("Healthy generalisation")

In [ ]:
impact = doctor.feature_impact(top_n=8)
print("Feature impact (score drop when removed):")
for feat, drop in sorted(impact.items(), key=lambda x: -x[1]):
    bar = chr(9608) * max(1, int(drop * 150))
    print(f"  {feat:20s}  {drop:.4f}  {bar}")

In [ ]:
improvement_report = doctor.improve()
actions = doctor.recommend_actions()
for i, a in enumerate(actions[:5], 1):
    print(f"{i}. {a}")

In [ ]:
trust = doctor.trust_score()
print(f"Trust Score: {trust.score:.0f} / 100")
status = "PRODUCTION READY" if trust.score >= 75 else ("Needs work" if trust.score >= 60 else "NOT ready")
print(status)

In [ ]:
confidence = doctor.pipeline_confidence()
print(f"Pipeline Confidence: {confidence} / 100")

In [ ]:
report_path = doctor.report(output_path="intermediate_report.html")
from IPython.display import IFrame, display
display(IFrame(src="intermediate_report.html", width="100%", height="600px"))

## Before vs After: auto_improve()

In [ ]:
doctor2 = DataDoctor()
doctor2.load("https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv")
doctor2.fit(target="Survived")
result = doctor2.auto_improve(tune=False)
print(f"Score delta: {result.score_delta:+.4f}")

---
*KaizenStat v0.5.1 · [GitHub](https://github.com/kaizenstat-python/KaizenStat) · MIT License*